# MELD (dataset) + FLAN-T5 + Prompt Engineering

Este notebook prepara el dataset MELD, construye ventanas de contexto por diálogo y evalúa **prompt engineering** con **FLAN-T5** (sin entrenamiento adicional).

In [1]:
import kagglehub
import os
import pandas as pd

C:\Users\Haojie\Desktop\MacOS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("zaber666/meld-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Haojie\.cache\kagglehub\datasets\zaber666\meld-dataset\versions\1


In [3]:
# Cargar CSVs de texto
train_path = os.path.join(path+'/MELD-RAW/MELD.Raw/train/', "train_sent_emo.csv")
dev_path   = os.path.join(path+'/MELD-RAW/MELD.Raw/', "dev_sent_emo.csv")
test_path  = os.path.join(path+'/MELD-RAW/MELD.Raw/', "test_sent_emo.csv")

train = pd.read_csv(train_path, encoding="utf-8")
dev   = pd.read_csv(dev_path, encoding="utf-8")
test  = pd.read_csv(test_path, encoding="utf-8")

In [4]:
# Limpieza mínima de caracteres raros (apóstrofo 0x92)
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return s
    return s.replace('\x92', "")#

for df_ in (train, dev, test):
    df_["Utterance"] = df_["Utterance"].apply(clean_text)


In [5]:
# Unificar splits y quedarnos con las columnas relevantes

##%%
df = pd.concat(
    [
        train.assign(split="train"),
        dev.assign(split="dev"),
        test.assign(split="test"),
    ],
    ignore_index=True,
)

df = df[
    [
        "Dialogue_ID",
        "Utterance_ID",
        "Speaker",
        "Utterance",
        "Emotion",
        "Sentiment",
        "split",
    ]
]

print(df.head())
print(df.shape)


   Dialogue_ID  Utterance_ID          Speaker  \
0            0             0         Chandler   
1            0             1  The Interviewer   
2            0             2         Chandler   
3            0             3  The Interviewer   
4            0             4         Chandler   

                                           Utterance   Emotion Sentiment  \
0  also I was the point person on my companys tra...   neutral   neutral   
1                    You mustve had your hands full.   neutral   neutral   
2                            That I did. That I did.   neutral   neutral   
3       So lets talk a little bit about your duties.   neutral   neutral   
4                             My duties?  All right.  surprise  positive   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
(13708, 7)


In [6]:
# Construir estructura de diálogos ordenados

##%%
dialogues = (
    df.groupby("Dialogue_ID")
      .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))
      .to_dict()
)

# Ejemplo de diálogo
dialogues[0][:5]

C:\Users\Haojie\AppData\Local\Temp\ipykernel_36128\503415823.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values("Utterance_ID").to_dict("records"))


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'Monica',
  'Utterance': 'What?',
  'Emotion': 'surprise',
  'Sentiment': 'negative',
  'split': 'dev'}]

In [7]:
# Normalizar hablantes (SPEAKER_0, SPEAKER_1, ... por diálogo)

##%%
def normalize_speakers(dialogue):
    speakers = list({turn["Speaker"] for turn in dialogue})
    mapping = {spk: f"SPEAKER_{i}" for i, spk in enumerate(speakers)}
    for turn in dialogue:
        turn["Speaker_Norm"] = mapping[turn["Speaker"]]
    return dialogue

dialogues_norm = {
    d_id: normalize_speakers(dialog)
    for d_id, dialog in dialogues.items()
}

dialogues_norm[0][:5]


[{'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Chandler',
  'Utterance': 'also I was the point person on my companys transition from the KL-5 to GR-6 system.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_5'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Mark',
  'Utterance': 'Why do all youre coffee mugs have numbers on the bottom?',
  'Emotion': 'surprise',
  'Sentiment': 'positive',
  'split': 'test',
  'Speaker_Norm': 'SPEAKER_4'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 0,
  'Speaker': 'Phoebe',
  'Utterance': 'Oh my God, hes lost it. Hes totally lost it.',
  'Emotion': 'sadness',
  'Sentiment': 'negative',
  'split': 'dev',
  'Speaker_Norm': 'SPEAKER_3'},
 {'Dialogue_ID': 0,
  'Utterance_ID': 1,
  'Speaker': 'The Interviewer',
  'Utterance': 'You mustve had your hands full.',
  'Emotion': 'neutral',
  'Sentiment': 'neutral',
  'split': 'train',
  'Speaker_Norm': 'SPEAKER_1'},
 {'Dialogue_ID': 0,
  'Utterance_ID':

In [8]:
# Construir filas con ventana de contexto (por ejemplo 2 turnos anteriores)

##%%
def build_context_rows(dialogue, window=4):
    rows = []
    dialogue = sorted(dialogue, key=lambda x: x["Utterance_ID"])
    for i, turn in enumerate(dialogue):
        ctx = dialogue[max(0, i - window):i]
        ctx_text = " ".join(
            f'{t["Speaker_Norm"]}: {t["Utterance"]}' for t in ctx
        )
        rows.append(
            {
                "dialogue_id": turn["Dialogue_ID"],
                "utt_id": turn["Utterance_ID"],
                "speaker": turn["Speaker_Norm"],
                "context": ctx_text,
                "text": turn["Utterance"],
                "emotion": turn["Emotion"],
                "sentiment": turn["Sentiment"],
                "split": turn["split"],
            }
        )
    return rows

all_rows = []
for d_id, dialog in dialogues_norm.items():
    all_rows.extend(build_context_rows(dialog, window=4))

df_model = pd.DataFrame(all_rows)
df_model.head()

,dialogue_id,utt_id,speaker,context,text,emotion,sentiment,split
0,0,0,SPEAKER_5,,also I was the point person on my companys tra...,neutral,neutral,train
1,0,0,SPEAKER_4,SPEAKER_5: also I was the point person on my c...,Why do all youre coffee mugs have numbers on t...,surprise,positive,test
2,0,0,SPEAKER_3,SPEAKER_5: also I was the point person on my c...,"Oh my God, hes lost it. Hes totally lost it.",sadness,negative,dev
3,0,1,SPEAKER_1,SPEAKER_5: also I was the point person on my c...,You mustve had your hands full.,neutral,neutral,train
4,0,1,SPEAKER_0,SPEAKER_5: also I was the point person on my c...,What?,surprise,negative,dev


In [9]:
# Construir el campo final para el prompt: contexto + [SEP] + turno actual

##%%
df_model["context"] = df_model["context"].fillna("").astype(str)

df_model["input_text"] = (
    "Context:\n" + df_model["context"].str.strip() +
    "\n\nCurrent utterance:\n" + df_model["text"].astype(str).str.strip()
)


df_model[["dialogue_id", "utt_id", "speaker", "input_text", "emotion", "split"]].head(10)


,dialogue_id,utt_id,speaker,input_text,emotion,split
0,0,0,SPEAKER_5,Context:\n\n\nCurrent utterance:\nalso I was t...,neutral,train
1,0,0,SPEAKER_4,Context:\nSPEAKER_5: also I was the point pers...,surprise,test
2,0,0,SPEAKER_3,Context:\nSPEAKER_5: also I was the point pers...,sadness,dev
3,0,1,SPEAKER_1,Context:\nSPEAKER_5: also I was the point pers...,neutral,train
4,0,1,SPEAKER_0,Context:\nSPEAKER_5: also I was the point pers...,surprise,dev
5,0,1,SPEAKER_2,Context:\nSPEAKER_4: Why do all youre coffee m...,anger,test
6,0,2,SPEAKER_2,"Context:\nSPEAKER_3: Oh my God, hes lost it. H...",neutral,test
7,0,2,SPEAKER_5,Context:\nSPEAKER_1: You mustve had your hands...,neutral,train
8,0,3,SPEAKER_1,Context:\nSPEAKER_0: What? SPEAKER_2: Oh. That...,neutral,train
9,0,4,SPEAKER_5,Context:\nSPEAKER_2: Oh. Thats so Monica can k...,surprise,train


In [10]:
# Lista de emociones (clases) presentes en MELD

##%%
emotions = sorted(df_model["emotion"].unique())
print("Emociones:", emotions)

# Separar splits (útil para evaluar solo en test)
train_df = df_model[df_model["split"] == "train"].reset_index(drop=True)
dev_df   = df_model[df_model["split"] == "dev"].reset_index(drop=True)
test_df  = df_model[df_model["split"] == "test"].reset_index(drop=True)

print(train_df["emotion"].value_counts())

Emociones: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64


# Prompt engineering (baseline)

Esta sección evalúa **FLAN‑T5** sin entrenamiento usando *few-shot prompting* y *label scoring* (se puntúa cada etiqueta candidata y se elige la de menor pérdida).

> Nota: este baseline sirve como comparación. Para mejores métricas, usa la sección de **fine‑tuning** más abajo.


In [11]:
import torch
import random
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# Baseline: FLAN-T5 sin fine-tuning (prompt engineering)
# =========================

MODEL_NAME = "google/flan-t5-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.truncation_side = "left"   # conserva el final (donde está el Current utterance)
MAX_INPUT_LEN = 512

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Etiquetas (MELD 7-way)
LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABELS_STR = ", ".join(LABELS)

# Ajustes few-shot
SHOTS_PER_LABEL = 1
SEED = 42

def build_fewshot(train_df, shots_per_label=SHOTS_PER_LABEL, seed=SEED):
    """
    Few-shot balanceado (mismos shots por clase).
    Importante: usar ejemplos cortos para evitar truncation.
    """
    rnd = random.Random(seed)
    blocks = []
    for lab in LABELS:
        subset = train_df[train_df["emotion"].str.lower() == lab]
        if subset.empty:
            continue
        n = min(shots_per_label, len(subset))
        # muestreo sin reemplazo para variedad
        idxs = rnd.sample(range(len(subset)), n)
        for j in idxs:
            row = subset.iloc[j]
            # ejemplo corto: solo turno actual (no contexto completo)
            x = f'{row["speaker"]}: {row["text"]}'.strip()
            blocks.append(f"Text: {x}\nLabel: {lab}\n")
    return "\n".join(blocks).strip()

FEWSHOT = build_fewshot(train_df)

def build_prompt(input_text: str) -> str:
    return (
        "You are a dialogue emotion classifier.\n"
        f"Choose exactly ONE label from: {LABELS_STR}\n"
        "Return only the label (one word), nothing else.\n\n"
        f"{FEWSHOT}\n\n"
        "Now classify the following dialogue turn.\n"
        f"{input_text}\n"
        "Label:"
    )

# Pre-tokenización de etiquetas (para acelerar el scoring)
pad_id = tok.pad_token_id
_label_ids = tok(LABELS, return_tensors="pt", padding=True, add_special_tokens=False).input_ids.to(device)

@torch.inference_mode()
def score_labels(prompt: str):
    """
    Devuelve una lista de pérdidas promedio por token (menor = mejor) para cada etiqueta candidata.
    """
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)

    B = _label_ids.size(0)
    # replicar el prompt para cada etiqueta
    input_ids = enc.input_ids.repeat(B, 1)
    attention_mask = enc.attention_mask.repeat(B, 1)

    labels = _label_ids

    # forward teacher-forcing: logits para cada etiqueta
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    logits = out.logits  # (B, T, V)

    # loss por token (ignorando padding)
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    loss_tok = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        reduction="none",
        ignore_index=pad_id,
    ).view(B, -1)

    denom = (shift_labels != pad_id).sum(dim=1).clamp(min=1)
    loss_per_label = loss_tok.sum(dim=1) / denom
    return loss_per_label.detach().cpu().tolist()

@torch.inference_mode()
def predict_one(input_text: str) -> str:
    prompt = build_prompt(input_text)
    losses = score_labels(prompt)
    best_i = int(min(range(len(losses)), key=lambda i: losses[i]))
    return LABELS[best_i]

def predict_batch(texts: list[str]) -> list[str]:
    return [predict_one(t) for t in texts]


Device: cuda


In [12]:
# Evaluación en el conjunto de test (muestra)
sample_size = 2000
test_sample = test_df.sample(n=min(sample_size, len(test_df)), random_state=42)

texts = test_sample["input_text"].tolist()
y_true = test_sample["emotion"].str.lower().tolist()
y_pred = predict_batch(texts)

print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=LABELS,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

       anger       0.14      1.00      0.24       270
     disgust       0.00      0.00      0.00        47
        fear       0.00      0.00      0.00        35
         joy       0.00      0.00      0.00       301
     neutral       0.00      0.00      0.00       982
     sadness       0.00      0.00      0.00       157
    surprise       0.00      0.00      0.00       208

    accuracy                           0.14      2000
   macro avg       0.02      0.14      0.03      2000
weighted avg       0.02      0.14      0.03      2000



# Fine-tuning de FLAN‑T5 (recomendado)

Esta sección hace **fine‑tuning supervisado** para que FLAN‑T5 aprenda a **generar directamente** la etiqueta.

Incluye:
- **Oversampling** (para clases minoritarias) y *downsampling* suave de `neutral`.
- Entrenamiento con **Early Stopping** (se detiene si no mejora en *dev*).
- Configuración pensada para **GPU 16GB**.

> Consejo: empieza con estos hiperparámetros. Si tienes OOM, baja `BATCH_SIZE` o `MAX_SOURCE_LEN`.


In [13]:
# Si no lo tienes instalado:
# !pip -q install -U transformers datasets accelerate sentencepiece evaluate peft

import pandas as pd
from datasets import Dataset

SEED = 42
LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

def make_source(input_text: str) -> str:
    return (
        "Classify the emotion of the Current utterance given the Context.\n"
        f"Return exactly one label from: {', '.join(LABELS)}.\n\n"
        f"{input_text}\n\n"
        "Label:"
    )

# ====== Oversampling / balancing ======
USE_OVERSAMPLING = True

# multiplica minoritarias (ajustable)
OVERSAMPLE_MULTIPLIER = {
    "fear": 6,
    "disgust": 6,
    "sadness": 3,
    # otras en 1 por defecto
}

# downsample suave de neutral para que no domine (ajusta 0.6–1.0)
NEUTRAL_KEEP_FRAC = 0.75

train_tmp = train_df[["input_text", "emotion"]].copy()
dev_tmp   = dev_df[["input_text", "emotion"]].copy()
test_tmp  = test_df[["input_text", "emotion"]].copy()

for d in (train_tmp, dev_tmp, test_tmp):
    d["emotion"] = d["emotion"].str.lower().str.strip()

print("Distribución TRAIN (antes):")
print(train_tmp["emotion"].value_counts())

if USE_OVERSAMPLING:
    parts = []
    for lab in LABELS:
        sub = train_tmp[train_tmp["emotion"] == lab]
        if sub.empty:
            continue

        # downsample neutral (opcional)
        if lab == "neutral" and NEUTRAL_KEEP_FRAC < 1.0:
            sub = sub.sample(frac=NEUTRAL_KEEP_FRAC, random_state=SEED)

        mult = int(OVERSAMPLE_MULTIPLIER.get(lab, 1))
        if mult > 1:
            # concatenación + pequeño muestreo con reemplazo para variar el orden
            sub_rep = [sub]
            for k in range(mult - 1):
                sub_rep.append(sub.sample(n=len(sub), replace=True, random_state=SEED + 13*(k+1)))
            sub = pd.concat(sub_rep, ignore_index=True)

        parts.append(sub)

    train_tmp = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nDistribución TRAIN (después):")
print(train_tmp["emotion"].value_counts())

train_ds = Dataset.from_pandas(train_tmp, preserve_index=False)
dev_ds   = Dataset.from_pandas(dev_tmp, preserve_index=False)
test_ds  = Dataset.from_pandas(test_tmp, preserve_index=False)

len(train_ds), len(dev_ds), len(test_ds)


Distribución TRAIN (antes):
emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64

Distribución TRAIN (después):
emotion
neutral     3532
sadness     2049
joy         1743
disgust     1626
fear        1608
surprise    1205
anger       1109
Name: count, dtype: int64


(12872, 1109, 2610)

In [16]:
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
)

FT_MODEL_NAME = "google/flan-t5-base"

tok_ft = AutoTokenizer.from_pretrained(FT_MODEL_NAME)
tok_ft.truncation_side = "left"  # conserva el final (donde suele estar el Current utterance)

model_ft = AutoModelForSeq2SeqLM.from_pretrained(FT_MODEL_NAME)

# 16GB GPU: 512 suele ir bien; si OOM prueba 384
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 8  # etiquetas cortas

def preprocess(batch):
    sources = [make_source(t) for t in batch["input_text"]]
    targets = batch["emotion"]
    model_inputs = tok_ft(sources, max_length=MAX_SOURCE_LEN, truncation=True)

    # tokeniza targets
    with tok_ft.as_target_tokenizer():
        labels = tok_ft(targets, max_length=MAX_TARGET_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
dev_tok   = dev_ds.map(preprocess, batched=True, remove_columns=dev_ds.column_names)
test_tok  = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tok_ft, model=model_ft)


Map:   0%|          | 0/12872 [00:00<?, ? examples/s]C:\Users\Haojie\Desktop\MacOS\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 2610/2610 [00:00<00:00, 7507.95 examples/s]


In [24]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, classification_report

def norm_label(s: str) -> str:
    s = (s or "").strip().lower()
    s = s.split()[0] if s else ""
    s = s.strip(".,:;!?'\"()[]{}")
    return s if s in LABELS else "unknown"

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    pred_text = tok_ft.batch_decode(preds, skip_special_tokens=True)

    # labels vienen con -100 en padding -> reemplaza por pad_token_id para decodificar
    labels = np.where(labels != -100, labels, tok_ft.pad_token_id)
    label_text = tok_ft.batch_decode(labels, skip_special_tokens=True)

    y_pred = [norm_label(x) for x in pred_text]
    y_true = [norm_label(x) for x in label_text]

    # métricas
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", labels=LABELS)
    f1w = f1_score(y_true, y_pred, average="weighted", labels=LABELS)
    unk = sum(1 for x in y_pred if x == "unknown") / max(1, len(y_pred))
    return {"accuracy": acc, "f1_macro": f1m, "f1_weighted": f1w, "unknown_rate": unk}

# ====== Configuración para GPU 16GB ======
BATCH_SIZE = 8
GRAD_ACCUM = 2  # batch efectivo = 16
NUM_EPOCHS = 6
LR = 1e-5  # estable para full fine-tuning

training_args = Seq2SeqTrainingArguments(
    output_dir="flan_meld_finetuned_balanced",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=8,
    generation_num_beams=1,

    warmup_ratio=0.06,
    weight_decay=0.01,
    label_smoothing_factor=0.1,

    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,

    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
    tokenizer=tok_ft,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
# Evaluación final en TEST
pred = trainer.predict(test_tok)
pred_text = tok_ft.batch_decode(pred.predictions, skip_special_tokens=True)

y_true = test_tmp["emotion"].tolist()
y_pred = [norm_label(x) for x in pred_text]

print(classification_report(y_true, y_pred, labels=LABELS, target_names=LABELS, zero_division=0))


## Notas

- Este notebook evalúa **prompt engineering** con **FLAN-T5 (encoder–decoder)**, sin fine-tuning.
- Para intentar mejorar macro-F1 sin entrenar:
  - incrementa `SHOTS_PER_LABEL` (p. ej. 3–4),
  - aumenta `MAX_INPUT_LEN` (p. ej. 512),
  - incrementa la ventana de contexto en `build_context_rows(window=...)`.
